# 🔭 **Explore the collection**

What is in this collection, when it was collected and what it looks like.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src import eda
from src.collection import Collection

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

&nbsp;

### 📂 **Open collection**

In [ ]:
collection = Collection()

&nbsp;

### 🖼️ **Load all images**

`add_features` adds the extended properties: color group, lightness, local time, megapixels, aspect, folder, ...

In [ ]:
images = eda.add_features(collection.images_df())
images[eda.EXTENDED].head(3)

&nbsp;

### 🩺 **Data audit**

#### 🚫 Null rate and value range of every column

In [ ]:
eda.audit(images)

#### ⭐ Rated and 🏷️ tagged

In [ ]:
print(f"Rated:  {images['rated'].mean():.0%}")  # 0 = unrated
print(f"Tagged: {(images['n_tags'] > 0).mean():.0%}")

#### 🕑 Suspicious timestamps

Many files in 1 minute = bulk copy or fallback time

In [ ]:
print("Created range:", images["created_dt"].min(), "->", images["created_dt"].max())
images["created_dt"].dt.floor("min").value_counts().head(3)

&nbsp;

### 🧬 **Extended properties**

#### 📂 Top source folders

In [ ]:
images["folder"].value_counts().head(5)[::-1].plot.barh()

&nbsp;

### 📊 **Basic**

#### 📄 Formats: most common

In [ ]:
sns.countplot(images, y="format", order=images["format"].value_counts().index)

#### 📄 Formats: share of files vs share of disk

In [ ]:
eda.share_files_vs_disk(images, "format").plot.bar()
plt.ylabel("%")

#### 📄 Formats: bytes per pixel

In [ ]:
sns.boxplot(images, x="format", y="bpp", log_scale=True)

#### ⏹️ Orientation

In [ ]:
sns.countplot(images, x="orientation")

#### 🖥️ Resolution tier

In [ ]:
sns.countplot(images, x="tier", order=eda.TIERS)

#### 📐 Aspect ratio

In [ ]:
sns.countplot(images, y="aspect", order=images["aspect"].value_counts().index)

#### ⭐ Rating distribution

Rated images only.

In [ ]:
sns.countplot(images[images["rated"]], x="rating")

#### 🏷️ Top tags

In [ ]:
images["tag_list"].explode().value_counts().head(15)[::-1].plot.barh()

#### 📈 Collection growth

By file creation time (the import time is not stored).

In [ ]:
created = images["created_dt"].sort_values()
plt.plot(created, range(1, len(created) + 1))
plt.ylabel("images")

#### 🕑 When I collect images

Weekday x hour of the day (UTC).

In [ ]:
sns.heatmap(eda.hour_weekday(images), cmap="viridis")

#### Disk space taken

How much disk space the biggest files take.

In [ ]:
eda.pareto(images).plot(x="files_%", y="disk_%", legend=False, xlabel="% of files", ylabel="% of disk")

&nbsp;

### 🔗 **Relationships & Behavioral**

#### 🏷️ Tags co-occurrence

red = appear together more than by chance.

In [ ]:
sns.heatmap(eda.tag_lift(images), cmap="coolwarm", center=0)

#### 🤯 Too big images

In [ ]:
eda.contact_sheet(images.nlargest(5, "size"), caption="size")

#### 🤏 Too small images

In [ ]:
eda.contact_sheet(images.nsmallest(5, "size"), caption="size")

#### 🧓 Too old images

In [ ]:
eda.contact_sheet(images.nsmallest(5, "created"), caption="created_dt")

#### 🎨 Color group distribution

Group of the most dominant color. Low chroma colors are black / gray / white.

In [ ]:
sns.countplot(images, x="color_group", hue="color_group", palette=eda.PALETTE, legend=False, dodge=False)

#### 🎨 Color group by lightness

In [ ]:
pd.crosstab(images["color_group"], images["bucket"]).plot.bar(stacked=True, colormap="viridis")

#### ⭐ Rating correlation

In [ ]:
rated = images[images["rated"]]
sns.heatmap(rated[eda.NUMERIC].corr("spearman")[["rating"]], annot=True, cmap="coolwarm", center=0)

#### ⭐ Rating vs other properties

In [ ]:
images[images["rated"]].groupby("rating")[["megapixels", "lightness", "chroma", "n_tags"]].mean()

&nbsp;

### 🧠 **Embeddings**

#### Load embeddings

In [ ]:
emb, X = eda.load_embeddings(collection, images)
print(f"{len(emb)}/{len(images)} images have embeddings")

#### Near-duplicates

Exact duplicates are already refused by hash, these are *visually* the same.

In [ ]:
pairs = eda.near_duplicates(emb, X, thres=0.95)
print(f"{len(pairs)} near-duplicate pairs")
eda.show_pairs(pairs)

#### ✨ Taste profile

Predict rating from embedding, then show unrated images you may like.

In [ ]:
pred, score = eda.taste_profile(emb, X)
eda.contact_sheet(emb[~emb["rated"]].assign(pred=pred).nlargest(5, "pred"), caption="pred")

&nbsp;

### 🎉 **Fun part**

#### 🎨 The color of you

In [ ]:
counts = eda.color_groups(images).stack().value_counts()
print(f"The color of you: {counts.index[0]} ({counts.iloc[0] / counts.sum():.0%} of all dominant colors)")
eda.swatch(counts.index[0])

#### ✨ Eyeris Wrapped

**Spotify Wrapped** mimic.

In [ ]:
print(eda.wrapped(images))

#### 🍹 Collection barcode

In [ ]:
eda.barcode(images)